# Position Sizing and Risk Management

Learn how to manage position sizes and control risk in your trading strategies.

## Topics Covered

1. Fixed position sizing
2. Percentage-based sizing
3. Kelly Criterion
4. Risk-based sizing (ATR)
5. Portfolio heat management
6. Stop-loss and take-profit

In [ ]:
import backtrader as bt
import pandas as pd
import numpy as np

# Create sample data
dates = pd.date_range('2023-01-01', periods=250, freq='D')
prices = 100 + np.cumsum(np.random.randn(250) * 2)

data = pd.DataFrame({
    'open': prices + np.random.randn(250) * 0.5,
    'high': prices + abs(np.random.randn(250) * 1.5),
    'low': prices - abs(np.random.randn(250) * 1.5),
    'close': prices,
    'volume': np.random.randint(1000, 10000, 250)
}, index=dates)

data.to_csv('sample_data_sizing.csv')
print("Sample data created")

## 1. Fixed Position Sizing

The simplest approach - always buy the same number of shares.

In [ ]:
class FixedSizeStrategy(bt.Strategy):
    """Strategy with fixed position size."""
    
    params = (('size', 10),)  # Always buy 10 shares
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=20)
    
    def next(self):
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                self.buy(size=self.p.size)
        else:
            if self.data.close[0] < self.sma[0]:
                self.close()

# Test fixed sizing
cerebro = bt.Cerebro()
data_feed = bt.feeds.GenericCSVData(
    dataname='sample_data_sizing.csv',
    datetime=0, open=1, high=2, low=3, close=4, volume=5,
    dtformat='%Y-%m-%d'
)
cerebro.adddata(data_feed)
cerebro.addstrategy(FixedSizeStrategy)
cerebro.broker.setcash(10000.0)

print(f'Starting Value: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'Final Value: {cerebro.broker.getvalue():.2f}')

## 2. Percentage-Based Sizing

Invest a fixed percentage of your portfolio in each trade.

In [ ]:
class PercentSizer(bt.Sizer):
    """Size positions as percentage of portfolio."""
    
    params = (('percent', 0.20),)  # 20% of portfolio
    
    def _getsizing(self, comminfo, cash, data, isbuy):
        if isbuy:
            # Calculate position size
            portfolio_value = self.broker.getvalue()
            position_value = portfolio_value * self.p.percent
            size = int(position_value / data.close[0])
            return size
        else:
            # Sell entire position
            return self.broker.getposition(data).size


class PercentStrategy(bt.Strategy):
    """Strategy using percentage-based sizing."""
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=20)
    
    def next(self):
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                self.buy()  # Sizer will determine size
        else:
            if self.data.close[0] < self.sma[0]:
                self.close()

# Test percentage sizing
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(PercentStrategy)
cerebro.addsizer(PercentSizer, percent=0.25)  # 25% per trade
cerebro.broker.setcash(10000.0)

print(f'Starting Value: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'Final Value: {cerebro.broker.getvalue():.2f}')

## 3. Kelly Criterion

Optimal position sizing based on win rate and win/loss ratio.

In [ ]:
class KellySizer(bt.Sizer):
    """Kelly Criterion position sizing.
    
    Kelly % = W - [(1-W) / R]
    W = Win rate
    R = Win/Loss ratio
    """
    
    params = (
        ('win_rate', 0.55),      # 55% win rate
        ('win_loss_ratio', 1.5), # Average win is 1.5x average loss
        ('kelly_fraction', 0.5), # Use half-Kelly for safety
    )
    
    def _getsizing(self, comminfo, cash, data, isbuy):
        if isbuy:
            # Calculate Kelly percentage
            W = self.p.win_rate
            R = self.p.win_loss_ratio
            kelly_pct = W - ((1 - W) / R)
            
            # Apply Kelly fraction (fractional Kelly)
            kelly_pct *= self.p.kelly_fraction
            
            # Ensure positive and reasonable
            kelly_pct = max(0.01, min(kelly_pct, 0.25))
            
            # Calculate position size
            portfolio_value = self.broker.getvalue()
            position_value = portfolio_value * kelly_pct
            size = int(position_value / data.close[0])
            
            return size
        else:
            return self.broker.getposition(data).size


# Test Kelly sizing
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(PercentStrategy)
cerebro.addsizer(KellySizer)
cerebro.broker.setcash(10000.0)

print(f'Starting Value: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'Final Value: {cerebro.broker.getvalue():.2f}')

## 4. ATR-Based Risk Sizing

Size positions based on volatility (Average True Range).

In [ ]:
class ATRSizer(bt.Sizer):
    """Position sizing based on ATR (volatility)."""
    
    params = (
        ('risk_percent', 0.02),  # Risk 2% per trade
        ('atr_mult', 2.0),       # Stop at 2x ATR
    )
    
    def __init__(self):
        self.atr = bt.indicators.ATR(self.strategy.data, period=14)
    
    def _getsizing(self, comminfo, cash, data, isbuy):
        if isbuy:
            # Calculate risk amount
            portfolio_value = self.broker.getvalue()
            risk_amount = portfolio_value * self.p.risk_percent
            
            # Calculate stop distance
            atr_value = self.atr[0]
            stop_distance = atr_value * self.p.atr_mult
            
            # Position size = Risk Amount / Stop Distance
            if stop_distance > 0:
                size = int(risk_amount / stop_distance)
                return max(1, size)  # At least 1 share
            return 1
        else:
            return self.broker.getposition(data).size


class ATRStrategy(bt.Strategy):
    """Strategy with ATR-based stops."""
    
    params = (('atr_mult', 2.0),)
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=20)
        self.atr = bt.indicators.ATR(period=14)
        self.order = None
        self.stop_price = None
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                self.order = self.buy()
                # Set stop-loss at entry - (ATR * multiplier)
                self.stop_price = self.data.close[0] - (self.atr[0] * self.p.atr_mult)
        else:
            # Check stop-loss
            if self.data.close[0] < self.stop_price:
                self.order = self.close()
            # Or exit on trend reversal
            elif self.data.close[0] < self.sma[0]:
                self.order = self.close()
    
    def notify_order(self, order):
        if order.status in [order.Completed]:
            self.order = None

# Test ATR sizing
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(ATRStrategy)
cerebro.addsizer(ATRSizer, risk_percent=0.02)
cerebro.broker.setcash(10000.0)

print(f'Starting Value: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'Final Value: {cerebro.broker.getvalue():.2f}')

## 5. Portfolio Heat Management

Limit total portfolio risk across all positions.

In [ ]:
class HeatSizer(bt.Sizer):
    """Limit total portfolio heat (risk exposure)."""
    
    params = (
        ('max_heat', 0.10),      # Max 10% total portfolio risk
        ('risk_per_trade', 0.02), # 2% risk per trade
    )
    
    def __init__(self):
        self.current_heat = 0.0
    
    def _getsizing(self, comminfo, cash, data, isbuy):
        if isbuy:
            # Check if we can add more risk
            if self.current_heat + self.p.risk_per_trade > self.p.max_heat:
                return 0  # No new positions
            
            # Calculate position size
            portfolio_value = self.broker.getvalue()
            position_value = portfolio_value * self.p.risk_per_trade
            size = int(position_value / data.close[0])
            
            # Update heat
            self.current_heat += self.p.risk_per_trade
            
            return size
        else:
            # Reduce heat when closing
            self.current_heat = max(0, self.current_heat - self.p.risk_per_trade)
            return self.broker.getposition(data).size

# Test heat management
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(PercentStrategy)
cerebro.addsizer(HeatSizer)
cerebro.broker.setcash(10000.0)

print(f'Starting Value: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'Final Value: {cerebro.broker.getvalue():.2f}')

## 6. Advanced: Stop-Loss and Take-Profit

Implement bracket orders with stops and targets.

In [ ]:
class BracketStrategy(bt.Strategy):
    """Strategy with stop-loss and take-profit orders."""
    
    params = (
        ('stop_loss', 0.02),    # 2% stop-loss
        ('take_profit', 0.05),  # 5% take-profit
    )
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=20)
        self.order = None
        self.stop_order = None
        self.limit_order = None
        self.entry_price = None
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                # Entry order
                self.order = self.buy()
        
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        
        if order.status == order.Completed:
            if order.isbuy():
                self.entry_price = order.executed.price
                
                # Create bracket orders
                stop_price = self.entry_price * (1 - self.p.stop_loss)
                limit_price = self.entry_price * (1 + self.p.take_profit)
                
                # Stop-loss order
                self.stop_order = self.sell(
                    exectype=bt.Order.Stop,
                    price=stop_price
                )
                
                # Take-profit order
                self.limit_order = self.sell(
                    exectype=bt.Order.Limit,
                    price=limit_price
                )
                
                print(f'BUY at {self.entry_price:.2f}')
                print(f'  Stop: {stop_price:.2f}')
                print(f'  Target: {limit_price:.2f}')
            
            elif order.issell():
                print(f'SELL at {order.executed.price:.2f}')
                
                # Cancel remaining bracket order
                if self.stop_order and self.stop_order.status != order.Completed:
                    self.cancel(self.stop_order)
                if self.limit_order and self.limit_order.status != order.Completed:
                    self.cancel(self.limit_order)
        
        self.order = None

# Test bracket orders
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(BracketStrategy)
cerebro.broker.setcash(10000.0)

print(f'Starting Value: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'Final Value: {cerebro.broker.getvalue():.2f}')

## Summary

You've learned:
- ✅ Fixed position sizing
- ✅ Percentage-based sizing
- ✅ Kelly Criterion optimization
- ✅ ATR-based risk management
- ✅ Portfolio heat control
- ✅ Stop-loss and take-profit orders

## Key Takeaways

1. **Never risk more than 1-2% per trade**
2. **Use volatility-based stops (ATR)**
3. **Limit total portfolio risk (heat)**
4. **Consider fractional Kelly for safety**
5. **Always use stop-losses**

## Next Steps

- **Tutorial 04**: Parameter Optimization
- **Tutorial 05**: Live Trading with CCXT